## 9. 분석 완료

### 생성된 파일
1. `park_service_coverage.csv` - 공원별 서비스 인구
2. `dong_service_score.csv` - 행정동별 서비스 수혜도
3. `sigungu_service_comparison.csv` - 시군구별 통계
4. `dong_service_score.geojson` - 행정동 GeoJSON (지도용)

### 다음 단계
- 시각화 및 대시보드 통합
- 공간 분포 지도 생성
- 취약 지역 식별 및 개선 방안 도출

In [ ]:
# 공원별 서비스 데이터
output_park = output_path / 'park_service_coverage.csv'
park_service_df.to_csv(output_park, index=False, encoding='utf-8-sig')
print(f"✓ 저장 완료: {output_park.name}")

# 행정동별 서비스 수혜도
output_dong = output_path / 'dong_service_score.csv'
dong_service_df.to_csv(output_dong, index=False, encoding='utf-8-sig')
print(f"✓ 저장 완료: {output_dong.name}")

# 시군구별 통계
output_sigungu = output_path / 'sigungu_service_comparison.csv'
sigungu_stats.to_csv(output_sigungu, index=False, encoding='utf-8-sig')
print(f"✓ 저장 완료: {output_sigungu.name}")

# 행정동 GeoJSON (지도 시각화용)
dong_service_gdf = dong_gdf.merge(
    dong_service_df[['ADM_NM', 'avg_service_per_capita', 'coverage_ratio', 'service_rank']],
    on='ADM_NM',
    how='left'
)
output_dong_geo = output_path / 'dong_service_score.geojson'
dong_service_gdf.to_file(output_dong_geo, driver='GeoJSON', encoding='utf-8')
print(f"✓ 저장 완료: {output_dong_geo.name}")

## 8. 결과 저장

In [ ]:
total_population = dong_service_df['total_population'].sum()
total_covered = dong_service_df['covered_population'].sum()
overall_coverage = (total_covered / total_population) * 100

print("=" * 80)
print("전체 통계 요약")
print("=" * 80)

print(f"\n【 전체 통계 】")
print(f"  - 총 인구: {total_population:,}명")
print(f"  - 500m 내 공원 접근 인구: {total_covered:,}명")
print(f"  - 전체 커버리지 비율: {overall_coverage:.2f}%")
print(f"  - 평균 1인당 서비스 면적: {dong_service_df['avg_service_per_capita'].mean():.2f} m²/인")
print(f"  - WHO 기준(9 m²/인) 충족 행정동: {len(dong_service_df[dong_service_df['avg_service_per_capita'] >= 9])}개 / {len(dong_service_df)}개")

print(f"\n【 서비스 수혜도 TOP 10 행정동 】")
top10_dong = dong_service_df.head(10)[
    ['service_rank', 'ADM_NM', 'SIGUNGU_NM', 'avg_service_per_capita', 'coverage_ratio']
]
display(top10_dong)

print(f"\n【 서비스 부족 BOTTOM 10 행정동 】")
bottom10_dong = dong_service_df.tail(10)[
    ['service_rank', 'ADM_NM', 'SIGUNGU_NM', 'avg_service_per_capita', 'coverage_ratio']
]
display(bottom10_dong)

## 7. 주요 통계 요약

In [ ]:
def gini_coefficient(x):
    """Gini 계수 계산"""
    x = np.array(x)
    x = x[x > 0]  # 0 제외
    if len(x) == 0:
        return 0
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * x)) / (n * np.sum(x)) - (n + 1) / n

gini_coef = gini_coefficient(dong_service_df['avg_service_per_capita'])

print(f"서비스 형평성 (Gini 계수): {gini_coef:.4f}")
print(f"  - 0: 완전 평등")
print(f"  - 1: 완전 불평등")
print(f"  - 현재: {'양호 (0.3 이하)' if gini_coef <= 0.3 else '보통 (0.3~0.5)' if gini_coef <= 0.5 else '불평등 (0.5 이상)'}")

## 6. 서비스 형평성 분석 (Gini 계수)

In [ ]:
# 시군구별 집계
sigungu_stats = dong_service_df.groupby('SIGUNGU_NM').agg({
    'total_population': 'sum',
    'covered_population': 'sum',
    'avg_service_per_capita': 'mean'
}).reset_index()

# 커버리지 비율 재계산
sigungu_stats['coverage_ratio'] = (
    sigungu_stats['covered_population'] / 
    sigungu_stats['total_population'] * 100
).round(2)

# 순위
sigungu_stats['rank'] = sigungu_stats['avg_service_per_capita'].rank(
    ascending=False
).astype(int)

sigungu_stats = sigungu_stats.sort_values('rank')

print("시군구별 통계:")
display(sigungu_stats)

## 5. 시군구별 집계

In [ ]:
# Step 3: 행정동별 서비스 점수 집계
print("Step 3: 행정동별 서비스 점수 집계 중...\n")
dong_results = []

for dong_idx, dong_name in enumerate(dong_gdf['ADM_NM'].unique()):
    if dong_idx % 50 == 0:
        print(f"  진행: {dong_idx}/{len(dong_gdf)} ({dong_idx/len(dong_gdf)*100:.1f}%)")
    
    dong_grids = grid_with_dong[grid_with_dong['ADM_NM'] == dong_name]
    total_pop = dong_grids['val'].sum()
    
    if total_pop == 0:
        continue
    
    service_score = 0
    covered_grids = set()
    
    for _, grid in dong_grids.iterrows():
        grid_centroid = grid.geometry.centroid
        grid_pop = grid['val']
        grid_id = grid['gid']
        
        # 격자 주변 공원 찾기
        for park_idx, park in park_gdf.iterrows():
            buffer = park.geometry.buffer(buffer_distance)
            
            if grid_centroid.within(buffer):
                # 공원의 공급-수요 비율 × 격자 인구
                R_j = park_supply_demand_ratio[park_idx]
                service_score += R_j * grid_pop
                
                # 커버된 격자 기록
                covered_grids.add(grid_id)
                break  # 가장 가까운 공원만 적용
    
    # 커버된 인구
    covered_pop = dong_grids[dong_grids['gid'].isin(covered_grids)]['val'].sum()
    
    # 1인당 평균
    avg_service = service_score / total_pop if total_pop > 0 else 0
    coverage_ratio = (covered_pop / total_pop) * 100 if total_pop > 0 else 0
    
    dong_results.append({
        'ADM_NM': dong_name,
        'SIGUNGU_NM': dong_grids['SIGUNGU_NM'].iloc[0] if len(dong_grids) > 0 else '',
        'total_population': int(total_pop),
        'covered_population': int(covered_pop),
        'coverage_ratio': round(coverage_ratio, 2),
        'total_service_score': round(service_score, 2),
        'avg_service_per_capita': round(avg_service, 2)
    })

print(f"\n✓ 완료: {len(dong_results)}개 행정동")

# DataFrame 생성
dong_service_df = pd.DataFrame(dong_results)

# 순위 계산
dong_service_df['service_rank'] = dong_service_df['avg_service_per_capita'].rank(
    ascending=False, method='min'
).astype(int)

dong_service_df = dong_service_df.sort_values('service_rank')

print("\n행정동별 서비스 수혜도 통계:")
dong_service_df.describe()

In [ ]:
# Step 2: 격자에 행정동 정보 추가
print("Step 2: 격자-행정동 매칭 중...")
grid_with_dong = gpd.sjoin(
    grid_gdf,
    dong_gdf[['ADM_NM', 'SIGUNGU_NM', 'geometry']],
    how='left',
    predicate='within'
)
print(f"✓ 완료: {len(grid_with_dong):,}개 격자 매칭")

grid_with_dong.head()

In [ ]:
# 버퍼 거리 설정 (500m)
buffer_distance = 500

# Step 1: 공원별 공급-수요 비율 계산
print("Step 1: 공원별 공급-수요 비율 계산 중...\n")
park_supply_demand_ratio = {}

for idx, park in park_gdf.iterrows():
    if idx % 100 == 0:
        print(f"  진행: {idx}/{len(park_gdf)} ({idx/len(park_gdf)*100:.1f}%)")
    
    park_geom = park.geometry
    park_area_m2 = park['df_park_area_km2'] * 1_000_000
    
    buffer = park_geom.buffer(buffer_distance)
    
    # 버퍼 내 격자
    grids_in_buffer = grid_gdf[grid_gdf.geometry.centroid.within(buffer)]
    total_demand = grids_in_buffer['val'].sum()
    
    if total_demand > 0:
        R_j = park_area_m2 / total_demand  # m²/인
    else:
        R_j = 0
    
    park_supply_demand_ratio[idx] = R_j

print(f"\n✓ 완료: {len(park_supply_demand_ratio)}개 공원")

## 4. 행정동별 서비스 수혜도 계산 (2SFCA 방법)

### 2SFCA (Two-Step Floating Catchment Area) 방법

**Step 1**: 각 공원의 공급-수요 비율 계산
$$R_j = \frac{S_j}{\sum_{k \in \{d_{jk} \leq d_0\}} P_k}$$

**Step 2**: 각 행정동의 서비스 접근성 점수 계산
$$A_i = \sum_{j \in \{d_{ij} \leq d_0\}} R_j$$

In [ ]:
# 500m 기준으로 압박도 분류
park_service_df['service_pressure'] = pd.cut(
    park_service_df['supply_per_capita_500m'],
    bins=[0, 5, 9, 15, float('inf')],
    labels=['과밀(0~5)', '부족(5~9)', '적정(9~15)', '여유(15+)']
)

print("공원 서비스 압박도 분포 (500m 기준):")
print(park_service_df['service_pressure'].value_counts())

print("\n서비스 인구 TOP 10 공원:")
top10_pop = park_service_df.nlargest(10, 'served_pop_500m')[
    ['park_name', 'park_area_km2', 'served_pop_500m', 'supply_per_capita_500m']
]
display(top10_pop)

print("\n1인당 면적 최하위 10 공원 (가장 혼잡):")
bottom10 = park_service_df[park_service_df['served_pop_500m'] > 0].nsmallest(
    10, 'supply_per_capita_500m'
)[['park_name', 'park_area_km2', 'served_pop_500m', 'supply_per_capita_500m']]
display(bottom10)

## 3. 공원 서비스 압박도 분석

1인당 서비스 면적을 기준으로 공원의 혼잡도를 분류합니다.

In [ ]:
# 버퍼 거리 설정 (미터)
buffer_distances = [300, 500, 1000]

park_service_results = []
total_parks = len(park_gdf)

print(f"총 {total_parks}개 공원 처리 시작...\n")

for idx, park in park_gdf.iterrows():
    if idx % 100 == 0:
        print(f"진행: {idx}/{total_parks} ({idx/total_parks*100:.1f}%)")
    
    park_geom = park.geometry
    park_area_km2 = park['df_park_area_km2']
    park_area_m2 = park_area_km2 * 1_000_000
    
    service_data = {
        'park_id': park.get('ID', f'PARK_{idx:04d}'),
        'park_name': park['LABEL'],
        'park_area_km2': park_area_km2,
        'park_area_m2': park_area_m2
    }
    
    # 각 버퍼 거리별 서비스 인구 계산
    for dist in buffer_distances:
        # 공원 경계 기준 버퍼 생성
        buffer = park_geom.buffer(dist)
        
        # 버퍼 내 격자 찾기 (격자 중심점 기준)
        grids_in_buffer = grid_gdf[grid_gdf.geometry.centroid.within(buffer)]
        
        # 인구 합계
        total_pop = grids_in_buffer['val'].sum()
        service_data[f'served_pop_{dist}m'] = int(total_pop)
        
        # 1인당 면적 (m²/인)
        if total_pop > 0:
            service_data[f'supply_per_capita_{dist}m'] = round(park_area_m2 / total_pop, 2)
        else:
            service_data[f'supply_per_capita_{dist}m'] = 0
    
    park_service_results.append(service_data)

print(f"\n✓ 완료: {total_parks}개 공원 처리")

# DataFrame 생성
park_service_df = pd.DataFrame(park_service_results)

print("\n공원별 서비스 인구 통계:")
park_service_df.describe()

## 2. 공원별 서비스 권역 인구 계산

각 공원의 경계(테두리)를 기준으로 버퍼를 확장하여 서비스 권역을 설정하고, 해당 권역 내 인구를 계산합니다.

In [ ]:
# 좌표계 통일
print("4. 좌표계 확인 및 통일 중...")
target_crs = 'EPSG:5179'

if grid_gdf.crs != target_crs:
    grid_gdf = grid_gdf.to_crs(target_crs)
if park_gdf.crs != target_crs:
    park_gdf = park_gdf.to_crs(target_crs)
if dong_gdf.crs != target_crs:
    dong_gdf = dong_gdf.to_crs(target_crs)

print(f"   ✓ 모든 데이터 좌표계 통일: {target_crs}")

In [ ]:
# 행정동 데이터
print("3. 행정동 데이터 로드 중...")
dong_gdf = gpd.read_file(base_path / 'seoul_dong_with_pop.geojson')
print(f"   ✓ 행정동 수: {len(dong_gdf):,}개")
print(f"   ✓ 총 인구: {dong_gdf['총인구(명)_전체'].sum():,.0f}명")

dong_gdf.head()

In [ ]:
# 공원 데이터
print("2. 공원 데이터 로드 중...")
park_gdf = gpd.read_file(base_path / 'seoul_parks.geojson')
print(f"   ✓ 공원 수: {len(park_gdf):,}개")
print(f"   ✓ 총 공원 면적: {park_gdf['df_park_area_km2'].sum():.2f} km²")

park_gdf.head()

In [ ]:
# 100m 격자 인구 데이터
print("1. 100m 격자 인구 데이터 로드 중...")
grid_gdf = gpd.read_file(base_path / 'seoul_100m_grid_population.shp')
print(f"   ✓ 격자 수: {len(grid_gdf):,}개")
print(f"   ✓ 컬럼: {list(grid_gdf.columns)}")
print(f"   ✓ 총 인구: {grid_gdf['val'].sum():,.0f}명")
print(f"   ✓ 좌표계: {grid_gdf.crs}")

grid_gdf.head()

In [ ]:
# 경로 설정
base_path = Path('../s1_data/s2_processed_data')
output_path = Path('../s1_data/s2_processed_data')

print(f"데이터 경로: {base_path}")

## 1. 데이터 로드

In [ ]:
# 필수 라이브러리 임포트
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")

# 공원 중심 서비스 권역 및 인구 커버리지 분석

## 목적
- 공원을 기준으로 서비스 권역 설정 (버퍼 300m/500m/1km)
- 각 공원이 수용하는 인구 파악
- 공원별 1인당 서비스 면적 계산
- 행정동/시군구별 공원 서비스 수혜도 분석 (2SFCA 방법)

## 학술적 근거
- 2SFCA (Two-Step Floating Catchment Area) 방법
- 공원 서비스 공급-수요 비율 분석